In [2]:
from src.database import create_database_manager
import pandas as pd
from sksurv.ensemble import RandomSurvivalForest
import numpy as np
from sksurv.metrics import concordance_index_censored

## 1) Load data

Using a single `load_query` since this feature table fits in memory (~90k rows).

If this gets bigger later, `Database.iterate_batches` in `database.py` can be used to stream in chunks.

In [3]:
from configuration.config import STUDY_END_DATE, FEATURE_SQL, DROP_COLS, RSF_PARAMS

# Centralized seed controls for stabilized evaluation
SEED_LIST = [13, 42, 67, 89, 123]
ACTIVE_SEED = SEED_LIST[0]
print(f"Seed list: {SEED_LIST}")
print(f"Active seed: {ACTIVE_SEED}")

datamanager = create_database_manager()
# Main path: single load into memory (~90k rows; minibatch not required for this dataset)
df = datamanager.load_query(FEATURE_SQL)
# Convert cutoff date used for snapshot features (t_pred = purchase_date)
df["t_pred_date"] = pd.to_datetime(df["t_pred_date"])
# Target-censoring baseline uses t_pred_date so duration is measured after prediction time
df["days_until_second_purchase"] = df["days_until_second_purchase"].fillna(
    (pd.Timestamp(STUDY_END_DATE) - df["t_pred_date"]).dt.days
)
print(DROP_COLS)

Seed list: [13, 42, 67, 89, 123]
Active seed: 13
['order_id', 'customer_id', 'order_status', 'purchase_date', 'order_approval_date', 'delivered_carrier_date', 'delivered_customer_date', 'estimated_delivery_date', 't_pred_date', 'has_second_purchase', 'days_until_second_purchase', 'customer_unique_id', 'customer_zip', 'first_review_date', 'latest_review_date', 'latest_shipping_limit_date', 'most_exp_product_id', 'most_exp_seller_id', 'most_exp_prod_category', 'most_exp_seller_zip', 'most_exp_seller_city', 'most_exp_seller_state', 'most_freq_category', 'val_seller_id', 'val_seller_zip', 'val_seller_city', 'val_seller_state', 'total_order_value', 'total_delivery_days']


## 2) Build train/val/test splits

Temporal ordering + class-ratio-aware splitting.

In [4]:
# Temporal sort baseline uses prediction cutoff timestamp (t_pred)
df['t_pred_date'] = pd.to_datetime(df['t_pred_date'])
df = df.sort_values(by='t_pred_date')

# split dataframe into censored and uncensored rows
censored_df = df[df['has_second_purchase'] == 0]
uncensored_df = df[df['has_second_purchase'] == 1]
# temp, test split ratios
temp_censored_ratio = int(len(censored_df) * 0.8)
temp_uncensored_ratio = int(len(uncensored_df) * 0.8)
# temp, test splits for censored entries
temp_censored = censored_df.iloc[:temp_censored_ratio]
test_censored = censored_df.iloc[temp_censored_ratio:]
# temp, test splits for uncensored entries
temp_uncensored = uncensored_df.iloc[:temp_uncensored_ratio]
test_uncensored = uncensored_df.iloc[temp_uncensored_ratio:]

# merge censored and uncensored dataframe for test portion
test_split = pd.concat([test_censored, test_uncensored])

# train, validation split ratios
train_censored_ratio = int(len(temp_censored) * (7/8))
train_uncensored_ratio = int(len(temp_uncensored) * (7/8))
# train, val split for censored entries
train_censored = temp_censored.iloc[:train_censored_ratio]
val_censored = temp_censored.iloc[train_censored_ratio:]
# train, val split for uncensored entries
train_uncensored = temp_uncensored.iloc[:train_uncensored_ratio]
val_uncensored = temp_uncensored.iloc[train_uncensored_ratio:]
# merge censored and uncensored dataframe for train, val portion
train_split = pd.concat([train_censored, train_uncensored])
val_split = pd.concat([val_censored, val_uncensored])


In [37]:

# ---------- quick sanity checks ----------
train_ids = set(train_split['order_id'])
val_ids = set(val_split['order_id'])
test_ids = set(test_split['order_id'])
all_ids = set(df['order_id'])

# overlap checks
overlap_train_val = len(train_ids & val_ids)
overlap_train_test = len(train_ids & test_ids)
overlap_val_test = len(val_ids & test_ids)
print(f"Overlaps (train/val, train/test, val/test): {overlap_train_val}, {overlap_train_test}, {overlap_val_test}")

# coverage + count checks
print(f"Split sizes -> train: {len(train_split)}, val: {len(val_split)}, test: {len(test_split)}")
print(f"Size check -> splits total: {len(train_split) + len(val_split) + len(test_split)}, df total: {len(df)}")
print(f"ID coverage check -> union IDs: {len(train_ids | val_ids | test_ids)}, df IDs: {len(all_ids)}")

# class imbalance checks
def _class_rate(split_df, name):
    event_rate = split_df['has_second_purchase'].mean()
    censored_rate = 1 - event_rate
    print(f"{name} rates -> event: {event_rate:.4f}, censored: {censored_rate:.4f}")

_class_rate(train_split, 'train')
_class_rate(val_split, 'val')
_class_rate(test_split, 'test')

# temporal boundary checks
print(f"Date ranges (t_pred) -> train: {train_split['t_pred_date'].min()} to {train_split['t_pred_date'].max()}")
print(f"Date ranges (t_pred) -> val:   {val_split['t_pred_date'].min()} to {val_split['t_pred_date'].max()}")
print(f"Date ranges (t_pred) -> test:  {test_split['t_pred_date'].min()} to {test_split['t_pred_date'].max()}")

print(
    "Boundary check (train <= val <= test) by t_pred:",
    train_split['t_pred_date'].max() <= val_split['t_pred_date'].min() <= test_split['t_pred_date'].min()
)



Overlaps (train/val, train/test, val/test): 0, 0, 0
Split sizes -> train: 65330, val: 9333, test: 18667
Size check -> splits total: 93330, df total: 93330
ID coverage check -> union IDs: 93330, df IDs: 93330
train rates -> event: 0.0159, censored: 0.9841
val rates -> event: 0.0160, censored: 0.9840
test rates -> event: 0.0160, censored: 0.9840
Date ranges (t_pred) -> train: 2016-10-25 00:00:00 to 2018-05-11 00:00:00
Date ranges (t_pred) -> val:   2018-01-22 00:00:00 to 2018-06-22 00:00:00
Date ranges (t_pred) -> test:  2018-03-09 00:00:00 to 2018-10-31 00:00:00
Boundary check (train <= val <= test) by t_pred: False


In [5]:
# split into target and training features (only drop columns that exist in the frame)
X_cols = df.drop([c for c in DROP_COLS if c in df.columns], axis=1).columns

X_train = train_split[X_cols]
X_validate = val_split[X_cols]
X_test = test_split[X_cols]

In [6]:
# create target component of form (event, duration)
train_array = np.array(
    list(zip(train_split["has_second_purchase"], train_split["days_until_second_purchase"])),
    dtype=[("has_second_purchase", bool), ("days_until_second_purchase", np.float32)],
)
validate_array_target = np.array(
    list(zip(val_split["has_second_purchase"], val_split["days_until_second_purchase"])),
    dtype=[("has_second_purchase", bool), ("days_until_second_purchase", np.float32)],
)
test_array_target = np.array(
    list(zip(test_split["has_second_purchase"], test_split["days_until_second_purchase"])),
    dtype=[("has_second_purchase", bool), ("days_until_second_purchase", np.float32)],
)

In [7]:
print(X_train.shape)
print(len(DROP_COLS))
print(f"Number of columns in X_train: {X_train.shape[1]}")
print(f"First 20 column names:\n{X_train.columns[:20].tolist()}")
print(f"Data types:\n{X_train.dtypes.value_counts()}")

(65330, 34)
29
Number of columns in X_train: 34
First 20 column names:
['purchase_month', 'purchase_day_of_week', 'purchased_on_weekend', 'most_freq_payment_type_encoded', 'payment_type_count', 'total_installments', 'total_payment_value', 'num_items_in_order', 'total_freight_value', 'total_merch_value', 'avg_price', 'price_std', 'min_price', 'price_range', 'freight_price_ratio', 'max_freight_ratio', 'min_freight_ratio', 'has_multiple_seller_states', 'num_seller_states', 'shipping_window_days']
Data types:
float64    18
int64      16
Name: count, dtype: int64


## 3) Feature setup + quick diagnostics

In [8]:
print(X_train.columns)

Index(['purchase_month', 'purchase_day_of_week', 'purchased_on_weekend',
       'most_freq_payment_type_encoded', 'payment_type_count',
       'total_installments', 'total_payment_value', 'num_items_in_order',
       'total_freight_value', 'total_merch_value', 'avg_price', 'price_std',
       'min_price', 'price_range', 'freight_price_ratio', 'max_freight_ratio',
       'min_freight_ratio', 'has_multiple_seller_states', 'num_seller_states',
       'shipping_window_days', 'most_exp_price', 'most_exp_freight',
       'most_exp_encoded_category', 'most_freq_encoded_category',
       'num_distinct_categories', 'most_freq_cat_concentration',
       'val_seller_encoded_state', 'has_duplicate_sellers',
       'has_multiple_sellers', 'num_distinct_sellers', 'avg_seller_price',
       'avg_seller_freight', 'seller_order_volume', 'seller_item_volume'],
      dtype='object')


## 4) Baseline model (untuned)

In [9]:
base_model = RandomSurvivalForest(n_jobs=-2, random_state=ACTIVE_SEED)
base_model.fit(X_train, train_array)
print(X_train.shape)

(65330, 34)


In [10]:
pred = base_model.predict(X_test)
print(concordance_index_censored(test_array_target['has_second_purchase'], test_array_target['days_until_second_purchase'], pred)[0])

0.4355255806299773


In [11]:
def permuter(
    model: RandomSurvivalForest, 
    X:pd.DataFrame, 
    Y:pd.DataFrame, 
    base_score:float, 
    n_repititions:int=5, 
    random_state:int=None) -> pd.Series:

    seed = ACTIVE_SEED if random_state is None else random_state
    rng = np.random.RandomState(seed)

    importance = {}

    for col in X.columns:
        running_c_index_avg = 0
        for _ in range(n_repititions):
            X_perm = X.copy()
            X_perm[col] = rng.permutation(X_perm[col].values)

            perm_prediction = model.predict(X_perm)
            perm_c_index = concordance_index_censored(Y['has_second_purchase'], Y['days_until_second_purchase'], perm_prediction)[0]
            running_c_index_avg += perm_c_index

        importance[col] = base_score - (running_c_index_avg / n_repititions)

    return pd.Series(importance).sort_values(ascending=False)

In [12]:
# Baseline score on validation + permutation importance ranking
base_predic = base_model.predict(X_validate)
base_c_index = concordance_index_censored(
    validate_array_target['has_second_purchase'],
    validate_array_target['days_until_second_purchase'],
    base_predic,
)[0]
perm_importances = permuter(base_model, X_validate, validate_array_target, base_c_index)

print("Base validation C-index:", base_c_index)
print("\nTop 20 permutation importances:")
print(perm_importances.head(20))
print("\nBottom 20 permutation importances:")
print(perm_importances.tail(20))

Base validation C-index: 0.28477031196434693

Top 20 permutation importances:
avg_price                         0.014145
total_installments                0.012915
most_exp_price                    0.010747
most_exp_freight                  0.008644
most_freq_payment_type_encoded    0.003004
has_duplicate_sellers             0.002896
most_exp_encoded_category         0.001601
num_items_in_order                0.001068
has_multiple_sellers              0.000597
max_freight_ratio                 0.000474
payment_type_count                0.000426
min_freight_ratio                 0.000418
purchase_day_of_week              0.000333
most_freq_cat_concentration       0.000329
num_distinct_sellers              0.000192
has_multiple_seller_states        0.000056
num_seller_states                 0.000041
shipping_window_days              0.000000
most_freq_encoded_category       -0.000152
avg_seller_price                 -0.000324
dtype: float64

Bottom 20 permutation importances:
num_distinc

In [13]:
# Auto-select features with materially negative permutation contribution
permuted_drop_cols = [col for col, val in perm_importances.items() if val < -0.005]
print("Auto-dropped feature count:", len(permuted_drop_cols))
print(permuted_drop_cols)

Auto-dropped feature count: 5
['avg_seller_freight', 'total_merch_value', 'min_price', 'freight_price_ratio', 'purchase_month']


In [14]:
# Drop poor performing features     
X_train_reduced = X_train.drop(permuted_drop_cols, axis=1)
X_val_reduced = X_validate.drop(permuted_drop_cols, axis=1)
X_test_reduced = X_test.drop(permuted_drop_cols, axis=1)

In [15]:
mod = RandomSurvivalForest(n_jobs=-2, random_state=ACTIVE_SEED)
mod.fit(X_train_reduced, train_array)
pred = mod.predict(X_val_reduced)
print(concordance_index_censored(validate_array_target['has_second_purchase'], validate_array_target['days_until_second_purchase'], pred)[0])


0.5311292995086276


In [37]:
# Custom scorer for GridSearchCV
def c_index_scorer(estimator, X, y):
    prediction = estimator.predict(X)
    return concordance_index_censored(y['has_second_purchase'], y['days_until_second_purchase'], prediction)[0]

In [24]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# hyperparameter tuning with event-stratified CV folds
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, 15],
    'min_samples_leaf': [50, 75, 100]
}

# Build folds from event indicator to reduce all-censored fold failures
cv_event_labels = train_array['has_second_purchase'].astype(int)
strat_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=ACTIVE_SEED)
cv_splits = list(strat_cv.split(X_train_reduced, cv_event_labels))

grid_search = GridSearchCV(
    estimator=RandomSurvivalForest(n_jobs=-2, random_state=ACTIVE_SEED),
    param_grid=param_grid,
    scoring=c_index_scorer,
    n_jobs=-2,
    cv=cv_splits,
    error_score='raise'
)
grid_search.fit(X_train_reduced, train_array)
tuned_model = grid_search.best_estimator_
print(grid_search.best_params_)

c:\Users\jocac\Projects\employee-churn-prediction\.employee_churn_venv\Lib\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
18 fits failed out of a total of 90.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
18 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\jocac\Projects\employee-churn-prediction\.employee_churn_venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\jocac\Projects\employee-churn-prediction\.employee_churn_venv\Lib\site-packages\sksurv\ensemble\forest.py", line 111, in fit
    event, time = check_ar

{'max_depth': 5, 'min_samples_leaf': 50, 'n_estimators': 100}


In [38]:
# Fit tuned model on full training set
# (using params selected from grid search)
params = {'max_depth': 15, 'min_samples_leaf': 75, 'n_estimators': 200}
tuned_model = RandomSurvivalForest(n_jobs=-2, random_state=ACTIVE_SEED, **params)
tuned_model.fit(X_train_reduced, train_array)


,n_estimators,200
,max_depth,15
,min_samples_split,6
,min_samples_leaf,75
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,bootstrap,True
,oob_score,False
,n_jobs,-2
,random_state,13


In [39]:
test_pred = tuned_model.predict(X_test_reduced)
test_c_index = concordance_index_censored(
    test_array_target['has_second_purchase'],
    test_array_target['days_until_second_purchase'],
    test_pred,
)[0]
print("Final test C-index:", test_c_index)

0.5191357916702429


In [ ]:
# Stabilized evaluation across multiple seeds
seed_eval_rows = []

for seed in SEED_LIST:
    seed_model = RandomSurvivalForest(n_jobs=-2, random_state=seed, **params)
    seed_model.fit(X_train_reduced, train_array)

    seed_test_pred = seed_model.predict(X_test_reduced)

    seed_test_c = concordance_index_censored(
        test_array_target['has_second_purchase'],
        test_array_target['days_until_second_purchase'],
        seed_test_pred,
    )[0]

    seed_eval_rows.append({
        'seed': seed,
        'test_c_index': seed_test_c
    })

seed_eval_df = pd.DataFrame(seed_eval_rows)
print(seed_eval_df)
print("Test mean/std:", round(seed_eval_df['test_c_index'].mean(), 4), round(seed_eval_df['test_c_index'].std(), 4))
print("Best test seed:", int(seed_eval_df.loc[seed_eval_df['test_c_index'].idxmax(), 'seed']))

   seed  test_c_index
0    13      0.519136
1    42      0.514527
2    67      0.517332
3    89      0.523174
4   123      0.520286


KeyError: 'val_c_index'